# 06 — Corrected report-to-label extraction and validation gate

This notebook first requires every frozen query × four conditions for all seven models and all three bundles, verifies each record against its run fingerprint, rejects empty outputs, and rereads the original paired JSON to require an explicit `labels` key. Before counting completeness, it backs up and removes only scientifically equivalent duplicate rows; duplicates that differ in reports, prompts, verifier outputs, labels, or provenance remain a hard failure. An existing but empty `labels` value is the all-zero 13-label reference vector.

It then resolves pinned official CheXbert code and checkpoint assets. Missing configured assets are installed automatically from the Stanford GitHub and StanfordAIMI Hugging Face repositories, the checkpoint checksum is verified, and the complete asset audit is saved. It also checks the Python interpreter used by CheXbert and installs the missing official `statsmodels==0.14.1` dependency when necessary. An isolated compatibility launcher prevents the installed Hugging Face `datasets` package from shadowing CheXbert's local `datasets/unlabeled_dataset.py` and restores the legacy `BertTokenizer.encode_plus` behavior. CheXbert's pre-tokenized token list is explicitly converted to one flat sequence using `cls_token_id` and `sep_token_id`, matching historical BERT formatting without relying on removed tokenizer helper methods. The launcher does not modify the official checkout or the installed Transformers package. Set `JAMIA_CHEXBERT_REPO` and `JAMIA_CHEXBERT_CHECKPOINT` to use existing installations, `JAMIA_CHEXBERT_AUTO_SETUP=0` to disable asset setup, or `JAMIA_CHEXBERT_AUTO_INSTALL_DEPS=0` to disable dependency installation.

The official command-line labeler preserves all 14 raw CheXbert states (including `No Finding`), applies the frozen binary mapping to the 13 evaluated observations, records script/checkpoint/input/output checksums, and executes frozen clinical sanity cases for negation, uncertainty, resolved findings, and support devices.

Finally, it freezes a ≥200-report sample stratified by source dataset, model, condition, and reference normal/abnormal status. Two independently shuffled annotation forms expose no model, condition, source label, or generation identifier; a separate crosswalk remains with the analyst. Publication metrics remain blocked until two distinct annotators complete the forms, all disagreements are adjudicated, all 13 labels have evaluable F1, and the prespecified macro-F1 threshold is met.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
# Never expose the implementation directory as a top-level import location:
# rerun_code/statistics.py would shadow Python's standard-library statistics.
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
clean_sys_path = []
for entry in sys.path:
    try:
        resolved_entry = Path(entry or ".").resolve()
    except Exception:
        resolved_entry = None
    if resolved_entry != implementation_dir:
        clean_sys_path.append(entry)
sys.path[:] = clean_sys_path
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code:", RERUN_DIR)
print("Output:", PATHS["root"])

In [ ]:
import importlib
import pandas as pd
from rerun_code.common import read_jsonl, write_json, write_jsonl
from rerun_code.config import sha256_path
from rerun_code.generation import CONDITIONS
import rerun_code.report_labeler as report_labeler_module
report_labeler_module = importlib.reload(report_labeler_module)
required_report_labeler_api = 8
actual_report_labeler_api = int(
    getattr(report_labeler_module, "REPORT_LABELER_API_VERSION", 0)
)
if actual_report_labeler_api < required_report_labeler_api:
    raise ImportError(
        "Notebook 06 and rerun_code/report_labeler.py are out of sync. "
        f"Required report-labeler API {required_report_labeler_api}, found "
        f"{actual_report_labeler_api} at "
        f"{Path(report_labeler_module.__file__).resolve()}. Copy the updated "
        "rerun_code/report_labeler.py to the active Biowulf re_run folder, "
        "restart the kernel, and rerun this cell."
    )
from rerun_code.report_labeler import audit_complete_generation_results, ensure_official_chexbert_assets, run_official_chexbert, map_chexbert_states, clinical_sanity_reports, validate_clinical_sanity_outputs, create_blinded_annotation_materials, create_adjudication_template, validate_two_annotators

generation, generation_audit = audit_complete_generation_results(
    generation_root=PATHS["generation"],
    bundles_root=PATHS["bundles"],
    model_keys=list(CONFIG["models"]),
    bundle_names=("mimic", "iuhn", "combined"),
    conditions=CONDITIONS,
    repair_equivalent_duplicates=True,
)
write_json(PATHS["labels"] / "generation_completeness_audit.json", generation_audit)
print("COMPLETE GENERATION COHORT VERIFIED:", generation_audit["n_records"], "records")
print(
    "Equivalent duplicate rows removed:",
    generation_audit["n_equivalent_duplicate_rows_removed"],
)
if generation_audit["duplicate_repair_backups"]:
    print(
        "Original JSONL backup files:",
        *generation_audit["duplicate_repair_backups"],
        sep="\n- ",
    )

sanity = clinical_sanity_reports()
label_input = pd.concat([
    generation[["generation_record_id", "final_report"]], sanity
], ignore_index=True)
chexbert_assets = ensure_official_chexbert_assets(
    repo=CONFIG["labeler"]["chexbert_repo"],
    checkpoint=CONFIG["labeler"]["checkpoint"],
)
write_json(PATHS["labels"] / "chexbert_asset_audit.json", chexbert_assets)
print("CheXbert code:", chexbert_assets["repository"])
print("CheXbert checkpoint:", chexbert_assets["checkpoint"])
labeler_dir = PATHS["labeler"] / "chexbert_full"
raw = run_official_chexbert(label_input, repo=chexbert_assets["repository"], checkpoint=chexbert_assets["checkpoint"], output_dir=labeler_dir, text_column="final_report")
vectors, states = map_chexbert_states(raw, CONFIG["labeler"]["uncertain_policy"])
n_generation = len(generation)
generation["prediction_vector"] = vectors[:n_generation]
generation["chexbert_states_14"] = states[:n_generation]
generation["unknown_json_labels"] = [[] for _ in range(n_generation)]
sanity_result = validate_clinical_sanity_outputs(
    states[n_generation:], vectors[n_generation:]
)
write_json(PATHS["labeler"] / "clinical_sanity_results.json", sanity_result)
write_jsonl(PATHS["labels"] / "labeled_generation.jsonl", generation.to_dict("records"))
write_json(PATHS["labels"] / "label_extraction_audit.json", {
    "generation_completeness_audit": str(PATHS["labels"] / "generation_completeness_audit.json"),
    "chexbert_provenance": str(labeler_dir / "chexbert_run_provenance.json"),
    "clinical_sanity_results": str(PATHS["labeler"] / "clinical_sanity_results.json"),
    "chexbert_asset_audit": str(PATHS["labels"] / "chexbert_asset_audit.json"),
    "uncertain_policy": CONFIG["labeler"]["uncertain_policy"],
    "evaluated_labels": list(CONFIG["labeler"]["uncertain_policy"]),
    "raw_chexbert_no_finding_preserved": True,
    "ground_truth_source": "paired_json.labels",
    "report_labeler_code": str(Path(report_labeler_module.__file__).resolve()),
    "report_labeler_code_sha256": sha256_path(Path(report_labeler_module.__file__).resolve()),
})

In [ ]:
validation_dir = PATHS["labeler"] / "validation"; validation_dir.mkdir(parents=True, exist_ok=True)
minimum = int(CONFIG["labeler"]["minimum_manual_validation_reports"])
annotation_manifest = create_blinded_annotation_materials(
    generation, validation_dir, n=minimum, seed=CONFIG["statistics"]["seed"]
)
annotator1 = Path(os.environ.get("JAMIA_ANNOTATOR1_CSV", validation_dir / "human_annotation_annotator1_completed.csv"))
annotator2 = Path(os.environ.get("JAMIA_ANNOTATOR2_CSV", validation_dir / "human_annotation_annotator2_completed.csv"))
adjudication_template = validation_dir / "human_adjudication_template.csv"
adjudicated = Path(os.environ.get("JAMIA_ADJUDICATED_CSV", validation_dir / "human_adjudication_completed.csv"))
gate = {
    "passed": False,
    "required_n": minimum,
    "required_macro_f1": CONFIG["labeler"]["minimum_macro_f1"],
    "required_annotators": CONFIG["labeler"]["required_annotators"],
    "required_f1_evaluable_labels": CONFIG["labeler"]["required_f1_evaluable_labels"],
    "annotation_manifest": annotation_manifest,
    "annotator1_completed": str(annotator1),
    "annotator2_completed": str(annotator2),
    "adjudication_completed": str(adjudicated),
}
if annotator1.exists() and annotator2.exists():
    try:
        adjudication_status = create_adjudication_template(
            annotator1, annotator2, adjudication_template
        )
        gate["adjudication"] = adjudication_status
        if adjudication_status["n_disagreement_cells"] == 0 or adjudicated.exists():
            result = validate_two_annotators(
                generation[["generation_record_id", "prediction_vector"]],
                annotator1_csv=annotator1,
                annotator2_csv=annotator2,
                crosswalk_csv=annotation_manifest["crosswalk"],
                adjudicated_csv=adjudicated if adjudicated.exists() else None,
            )
            gate.update(result)
            gate["passed"] = (
                result["n"] >= minimum
                and result["n_f1_evaluable_labels"] == CONFIG["labeler"]["required_f1_evaluable_labels"]
                and result["macro_f1"] is not None
                and result["macro_f1"] >= CONFIG["labeler"]["minimum_macro_f1"]
            )
            if not gate["passed"]:
                gate["pending"] = "The completed forms do not meet the prespecified validation thresholds."
        else:
            gate["pending"] = f"Complete adjudication template: {adjudication_template}"
    except Exception as exc:
        # Always replace a stale gate file with an explicit failed gate.
        # This makes crosswalk/hash/column problems visible to Notebook 07.
        gate["pending"] = "Annotation validation failed; correct the completed forms and rerun this cell."
        gate["validation_error"] = f"{type(exc).__name__}: {exc}"
else:
    gate["pending"] = "Two independently completed blinded annotation forms are required."
instructions_path = validation_dir / "HUMAN_ANNOTATION_NEXT_STEPS.txt"
instructions_path.write_text(
    "Notebook 06 human-validation checkpoint\n\n"
    f"Annotator 1 template: {annotation_manifest['annotator1_template']}\n"
    f"Annotator 2 template: {annotation_manifest['annotator2_template']}\n\n"
    f"Save completed annotator 1 form as: {annotator1}\n"
    f"Save completed annotator 2 form as: {annotator2}\n\n"
    "Each independent annotator must fill every human_* field with 0 or 1 and "
    "use one consistent, nonempty annotator_id. The two IDs must differ. Do not "
    "give either annotator the analyst-only crosswalk.\n\n"
    "After both forms are complete, rerun this cell. If disagreements exist, "
    f"complete {adjudication_template} with a third annotator and save the result "
    f"as {adjudicated}.\n",
    encoding="utf-8",
)
gate["instructions_file"] = str(instructions_path)
write_json(validation_dir / "validation_gate.json", gate)
print(json.dumps(gate, indent=2))
if gate["passed"]:
    print("LABELER VALIDATION GATE PASSED. Notebook 07 may now be run.")
else:
    print("\nHUMAN ANNOTATION PENDING — THIS IS A PLANNED CHECKPOINT, NOT A CODE ERROR.")
    print("Annotator 1 template:", annotation_manifest["annotator1_template"])
    print("Annotator 2 template:", annotation_manifest["annotator2_template"])
    print("Instructions:", instructions_path)
    print("Notebook 07 remains blocked until validation_gate.json has passed=true.")